In [1]:
import torch
import torch.nn as nn

# ──────────────────────────────────────────────
# 1. Single neuron with MANUAL backpropagation
# ──────────────────────────────────────────────

class NeuronManualBackprop:
    """
    A single sigmoid neuron with hand-coded forward & backward passes.

    Forward:  z = x·w + b  →  a = σ(z)
    Loss:     L = (a - y)²       (MSE for one sample)
    Backward: compute dL/dw, dL/db using the chain rule
    """

    def __init__(self, n_features: int):
        self.w = torch.randn(n_features) * 0.01   # weights
        self.b = torch.tensor(0.0)                  # bias

        # Cache for backward pass
        self.x = None       # input
        self.z = None       # pre-activation  (x·w + b)
        self.a = None       # activation      (σ(z))

    def sigmoid(self, z: torch.Tensor) -> torch.Tensor:
        return 1.0 / (1.0 + torch.exp(-z))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass: x → z → a"""
        self.x = x
        self.z = x @ self.w + self.b          # linear combination
        self.a = self.sigmoid(self.z)          # activation
        return self.a

    def backward(self, y: torch.Tensor) -> dict:
        """
        Backward pass — chain rule, step by step:

        L  = (a - y)²
        dL/da = 2(a - y)

        a  = σ(z)
        da/dz = σ(z) · (1 - σ(z))

        z  = x·w + b
        dz/dw = x
        dz/db = 1

        Chain rule:
        dL/dw = dL/da · da/dz · dz/dw
        dL/db = dL/da · da/dz · dz/db
        """
        # Step 1: loss gradient
        dL_da = 2.0 * (self.a - y)

        # Step 2: sigmoid derivative
        da_dz = self.a * (1.0 - self.a)

        # Step 3: combine into "delta"
        delta = dL_da * da_dz                  # dL/dz

        # Step 4: gradients for weights and bias
        dL_dw = delta * self.x                 # dL/dw = delta · x
        dL_db = delta                          # dL/db = delta · 1

        return {"dL_dw": dL_dw, "dL_db": dL_db, "delta": delta}

    def update(self, grads: dict, lr: float = 0.1):
        """Gradient descent step."""
        self.w -= lr * grads["dL_dw"]
        self.b -= lr * grads["dL_db"]


# ──────────────────────────────────────────────
# 2. Same neuron using PyTorch autograd
# ──────────────────────────────────────────────

class NeuronAutograd(nn.Module):
    """Same single neuron, letting PyTorch compute gradients."""

    def __init__(self, n_features: int):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid(self.linear(x))


# ──────────────────────────────────────────────
# 3. Step-by-step walkthrough
# ──────────────────────────────────────────────

def step_by_step_demo():
    """Trace every value through one forward + backward pass."""

    torch.manual_seed(42)

    x = torch.tensor([1.0, 2.0, 3.0])    # input
    y = torch.tensor(1.0)                  # target

    neuron = NeuronManualBackprop(n_features=3)
    neuron.w = torch.tensor([0.5, -0.3, 0.1])
    neuron.b = torch.tensor(0.2)

    print("── Step-by-step single backward pass ──\n")
    print(f"  Input x:    {x.tolist()}")
    print(f"  Target y:   {y.item()}")
    print(f"  Weights w:  {neuron.w.tolist()}")
    print(f"  Bias b:     {neuron.b.item()}\n")

    # Forward
    a = neuron.forward(x)
    z = neuron.z
    loss = (a - y) ** 2

    print(f"  FORWARD:")
    print(f"    z = x·w + b = {z.item():.4f}")
    print(f"    a = σ(z)    = {a.item():.4f}")
    print(f"    L = (a-y)²  = {loss.item():.4f}\n")

    # Backward
    grads = neuron.backward(y)

    print(f"  BACKWARD (chain rule):")
    print(f"    dL/da = 2(a-y)          = {2*(a-y).item():.4f}")
    print(f"    da/dz = σ(z)·(1-σ(z))  = {(a*(1-a)).item():.4f}")
    print(f"    delta = dL/da · da/dz   = {grads['delta'].item():.4f}")
    print(f"    dL/dw = delta · x       = {grads['dL_dw'].tolist()}")
    print(f"    dL/db = delta           = {grads['dL_db'].item():.4f}\n")


# ──────────────────────────────────────────────
# 4. Verify: manual gradients == autograd
# ──────────────────────────────────────────────

def verify_gradients():
    """Prove manual backprop matches PyTorch autograd exactly."""

    torch.manual_seed(7)
    x = torch.tensor([[1.0, 2.0, 3.0]])
    y = torch.tensor([[1.0]])

    # --- Manual ---
    manual = NeuronManualBackprop(3)
    manual.w = torch.tensor([0.5, -0.3, 0.1])
    manual.b = torch.tensor(0.2)

    a_manual = manual.forward(x.squeeze())
    grads    = manual.backward(y.squeeze())

    # --- Autograd ---
    auto = NeuronAutograd(3)
    auto.linear.weight.data = torch.tensor([[0.5, -0.3, 0.1]])
    auto.linear.bias.data   = torch.tensor([0.2])

    a_auto = auto(x)
    loss   = (a_auto - y) ** 2
    loss.backward()

    print("── Verification: manual == autograd ──\n")
    print(f"  Manual dL/dw: {grads['dL_dw'].tolist()}")
    print(f"  Autograd dw:  {auto.linear.weight.grad.squeeze().tolist()}")
    match_w = torch.allclose(grads["dL_dw"], auto.linear.weight.grad.squeeze(), atol=1e-5)

    print(f"  Manual dL/db: {grads['dL_db'].item():.6f}")
    print(f"  Autograd db:  {auto.linear.bias.grad.item():.6f}")
    match_b = torch.allclose(grads["dL_db"].unsqueeze(0), auto.linear.bias.grad, atol=1e-5)

    print(f"\n  Weights match: {match_w}")
    print(f"  Bias match:    {match_b}\n")


# ──────────────────────────────────────────────
# 5. Training comparison
# ──────────────────────────────────────────────

def training_demo():
    """Train both versions on a tiny dataset and compare convergence."""

    torch.manual_seed(0)

    # Tiny dataset
    X = torch.tensor([[0.5, 1.0], [1.5, 0.5], [3.0, 3.5], [2.5, 2.0]])
    y = torch.tensor([0.0, 0.0, 1.0, 1.0])

    LR = 0.5
    EPOCHS = 100

    # --- Manual training ---
    manual = NeuronManualBackprop(2)
    manual.w = torch.tensor([0.01, -0.01])
    manual.b = torch.tensor(0.0)

    print("── Training: manual backprop ──\n")
    for epoch in range(1, EPOCHS + 1):
        total_loss = 0.0
        for i in range(len(X)):
            a = manual.forward(X[i])
            loss = (a - y[i]) ** 2
            total_loss += loss.item()
            grads = manual.backward(y[i])
            manual.update(grads, lr=LR)

        if epoch % 25 == 0:
            preds = torch.stack([manual.forward(X[j]) for j in range(len(X))])
            acc = ((preds >= 0.5).float().squeeze() == y).float().mean()
            print(f"  Epoch {epoch:3d} | Loss: {total_loss/len(X):.4f} | Acc: {acc:.0%}")

    # --- Autograd training ---
    auto = NeuronAutograd(2)
    auto.linear.weight.data = torch.tensor([[0.01, -0.01]])
    auto.linear.bias.data   = torch.tensor([0.0])
    optimizer = torch.optim.SGD(auto.parameters(), lr=LR)

    print("\n── Training: PyTorch autograd ──\n")
    for epoch in range(1, EPOCHS + 1):
        total_loss = 0.0
        for i in range(len(X)):
            pred = auto(X[i].unsqueeze(0))
            loss = (pred - y[i]) ** 2
            total_loss += loss.item()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if epoch % 25 == 0:
            with torch.no_grad():
                preds = auto(X)
                acc = ((preds >= 0.5).float().squeeze() == y).float().mean()
            print(f"  Epoch {epoch:3d} | Loss: {total_loss/len(X):.4f} | Acc: {acc:.0%}")

    print()


# ──────────────────────────────────────────────
# 6. Run everything
# ──────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 55)
    print("  Backpropagation for a Single Neuron")
    print("=" * 55 + "\n")

    step_by_step_demo()
    verify_gradients()
    training_demo()

    print("── Chain rule recap ──")
    print("  dL/dw = dL/da · da/dz · dz/dw")
    print("        = 2(a-y) · σ(z)(1-σ(z)) · x")
    print("  That's ALL backprop is — the chain rule, applied layer by layer.")

  Backpropagation for a Single Neuron

── Step-by-step single backward pass ──

  Input x:    [1.0, 2.0, 3.0]
  Target y:   1.0
  Weights w:  [0.5, -0.30000001192092896, 0.10000000149011612]
  Bias b:     0.20000000298023224

  FORWARD:
    z = x·w + b = 0.4000
    a = σ(z)    = 0.5987
    L = (a-y)²  = 0.1611

  BACKWARD (chain rule):
    dL/da = 2(a-y)          = -0.8026
    da/dz = σ(z)·(1-σ(z))  = 0.2403
    delta = dL/da · da/dz   = -0.1928
    dL/dw = delta · x       = [-0.19283922016620636, -0.3856784403324127, -0.5785176753997803]
    dL/db = delta           = -0.1928

── Verification: manual == autograd ──

  Manual dL/dw: [-0.19283922016620636, -0.3856784403324127, -0.5785176753997803]
  Autograd dw:  [-0.19283920526504517, -0.38567841053009033, -0.5785176157951355]
  Manual dL/db: -0.192839
  Autograd db:  -0.192839

  Weights match: True
  Bias match:    True

── Training: manual backprop ──

  Epoch  25 | Loss: 0.0502 | Acc: 100%
  Epoch  50 | Loss: 0.0233 | Acc: 100%
  Ep